In [113]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

In [114]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

In [115]:

PATIENTS = [559, 563, 570, 575, 588, 591]
HORITZO = {30:6, 60:12}  # minuts : pas de 5 mins

In [116]:
# Funcio per obtenir els datasets
def load_data(id: int, train_or_test: str) -> pd.DataFrame:
    df=pd.read_csv(f'../data/{id}/{id}_{train_or_test}.csv', sep=';', header = None, names = cols)
    return df


df_559_train = load_data(559, 'train')
df_559_test = load_data(559, 'test')

In [117]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    prep = df.copy()

    prep['timestamp'] = pd.to_datetime(
        dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute)
    )
    
    prep.sort_values('timestamp', inplace=True)

    # Coma decimal a punt
    convert = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in convert:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem tipo
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")
    prep = pd.get_dummies(prep, columns=['meal_type'], dummy_na=False, prefix='meal')

    # Drop columnas amb casi tot NaN o valor constant
    prep = prep.drop(columns=["second","finger_stick","meal"])

    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)
    prep[invalid_zero] = prep[invalid_zero].fillna(method='ffill')

    prep = prep.dropna(subset=['glucose_level'])


    prep = prep.drop(columns=['timestamp'])
    return prep

In [118]:
df_559_train = preprocess(df_559_train)
df_559_test = preprocess(df_559_test)

display(df_559_train.tail())
display(df_559_test.head())
display(df_559_test.tail())

,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Correccion_hipo,meal_Desayuno,meal_Snack
12078,2022,1,17,23,35,161.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000213,92.3,89.6,0,94,False,False,False,False,False
12079,2022,1,17,23,40,164.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000201,92.3,89.6,0,94,False,False,False,False,False
12080,2022,1,17,23,45,168.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000198,92.3,89.6,0,94,False,False,False,False,False
12081,2022,1,17,23,50,172.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000192,92.3,89.6,0,94,False,False,False,False,False
12082,2022,1,17,23,55,176.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000188,92.3,89.6,0,94,False,False,False,False,False


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno,meal_Snack
0,2022,1,18,0,0,179.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000183,92.3,89.6,0,94,False,False,False,False
1,2022,1,18,0,5,183.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000182,92.3,89.6,0,94,False,False,False,False
2,2022,1,18,0,10,187.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000177,92.3,89.6,0,94,False,False,False,False
3,2022,1,18,0,15,191.0,0.83,0.0,3,0,0,0,0,0,56.0,0.000169,92.3,89.6,0,94,False,False,False,False
4,2022,1,18,0,20,195.0,0.83,0.0,3,0,0,0,0,0,56.0,0.000166,92.3,89.6,0,94,False,False,False,False


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno,meal_Snack
2871,2022,1,27,23,15,185.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2872,2022,1,27,23,20,183.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2873,2022,1,27,23,25,182.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2874,2022,1,27,23,30,180.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2875,2022,1,27,23,35,177.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False


In [119]:
print('Shape train: ', df_559_train.shape)
print('Shape test: ', df_559_test.shape)
print(df_559_train.isnull().mean()*100)

Shape train:  (12081, 25)
Shape test:  (2876, 24)
year                      0.000000
month                     0.000000
day                       0.000000
hour                      0.000000
minute                    0.000000
glucose_level             0.000000
basal                     0.000000
bolus                     0.000000
sleep                     0.000000
work                      0.000000
stressors                 0.000000
hypo_event                0.000000
illness                   0.000000
exercise                  0.000000
basis_heart_rate          1.158844
basis_gsr                 1.158844
basis_skin_temperature    1.158844
basis_air_temperature     1.158844
basis_step                0.000000
basis_sleep               0.000000
meal_Almuerzo             0.000000
meal_Cena                 0.000000
meal_Correccion_hipo      0.000000
meal_Desayuno             0.000000
meal_Snack                0.000000
dtype: float64


In [120]:
def make_xy(df: pd.DataFrame, steps: int):

    y = df['glucose_level'].shift(-steps)
    X = df.drop(columns=['glucose_level'])

    mask = y.notna()

    return X[mask], y[mask]

def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

In [121]:
# ------------------------------------------------
# Entrenamiento, predicción y métrica paciente por paciente
# ------------------------------------------------
all_metrics = []

for id in PATIENTS:
    print(f'\nPaciente {id}')
    train_raw = load_data(id, 'train')
    test_raw  = load_data(id, 'test')

    train = preprocess(train_raw).drop(columns=['meal_Correccion_hipo'])
    test  = preprocess(test_raw)

    # ignorar los primeros 60 min del test
    test = test.iloc[12:].reset_index(drop=True)

    display(train.head())
    display(test.head())

    for minuts, steps in HORITZO.items():
        # ---------- entrenamiento offline ----------
        X_train, y_train = make_xy(train, steps)

        display(y_train.info())

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        # ---------- predicción en test ----------
        X_test = test.iloc[:-steps].drop(columns=['glucose_level'])
        display(X_test.head())

        y_true = test['glucose_level'].iloc[:-steps].reset_index(drop=True)
        y_pred = model.predict(X_test)

        # guardar CSV requerido
        out_csv = f'../data/predicted/pred{id}_{minuts}min.csv'

        df_pred = pd.DataFrame({'idx_original': X_test.index, f'pred_glucose_t+{minuts}': y_pred})
        df_pred.to_csv(out_csv, index=False)

        # ---------- métricas ----------
        rmse, mae = evaluate(y_true, y_pred)
        all_metrics.append({
            'Paciente': id,
            'Horizonte': f'{minuts} min',
            'RMSE': rmse,
            'MAE' : mae
        })
        print(f'{minuts} min: RMSE={rmse:.2f}  MAE={mae:.2f}')

# ------------------------------------------------
# Tabla final (promedio incluido)
# ------------------------------------------------
metrics_df = pd.DataFrame(all_metrics)
prom = metrics_df.groupby('Horizonte')[['RMSE','MAE']].mean().reset_index()
prom.insert(0, 'Paciente', 'PROMEDIO')
result = pd.concat([metrics_df, prom], ignore_index=True)
print('\n==== RESULTADOS FINALES ====')
print(result.to_string(index=False))


Paciente 559


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno,meal_Snack
2,2021,12,7,1,15,101.0,0.65,0.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,0,0,False,False,False,False
3,2021,12,7,1,20,98.0,0.65,0.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,0,0,False,False,False,False
4,2021,12,7,1,25,104.0,0.65,0.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,0,0,False,False,False,False
5,2021,12,7,1,30,112.0,0.65,0.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,0,0,False,False,False,False
6,2021,12,7,1,35,120.0,0.65,0.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,0,0,False,False,False,False


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno,meal_Snack
0,2022,1,18,1,0,225.0,0.83,0.0,3,0,0,0,0,0,59.0,0.000319,91.22,89.60,0,94,False,False,False,False
1,2022,1,18,1,5,233.0,0.83,0.0,3,0,0,0,0,0,63.0,0.000521,91.40,90.32,0,94,False,False,False,False
2,2022,1,18,1,10,237.0,0.83,0.0,3,0,0,0,0,0,63.0,0.000921,92.30,90.50,0,94,False,False,False,False
3,2022,1,18,1,15,241.0,0.83,0.0,3,0,0,0,0,0,64.0,0.001542,92.30,90.50,0,94,False,False,False,False
4,2022,1,18,1,20,246.0,0.83,0.0,3,0,0,0,0,0,68.0,0.002376,92.30,90.50,0,94,False,False,False,False


<class 'pandas.core.series.Series'>
Index: 12075 entries, 2 to 12076
Series name: glucose_level
Non-Null Count  Dtype  
--------------  -----  
12075 non-null  float64
dtypes: float64(1)
memory usage: 188.7 KB


None

...

,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno
0,2021,10,29,1,0,205.0,1.5,0.0,3,0,0,0,0,0,78.0,0.000092,86.36,81.50,0,88,False,False,False
1,2021,10,29,1,5,204.0,1.5,0.0,3,0,0,0,0,0,78.0,0.000099,86.90,82.40,0,88,False,False,False
2,2021,10,29,1,10,203.0,1.5,0.0,3,0,0,0,0,0,78.0,0.000123,88.16,85.46,0,88,False,False,False
3,2021,10,29,1,15,204.0,1.5,0.0,3,0,0,0,0,0,77.0,0.000127,88.70,86.18,0,88,False,False,False
4,2021,10,29,1,20,203.0,1.5,0.0,3,0,0,0,0,0,77.0,0.000098,89.06,87.08,0,88,False,False,False


<class 'pandas.core.series.Series'>
Index: 13092 entries, 11 to 13102
Series name: glucose_level
Non-Null Count  Dtype  
--------------  -----  
13092 non-null  float64
dtypes: float64(1)
memory usage: 204.6 KB


None

,year,month,day,hour,minute,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno
0,2021,10,29,1,0,1.5,0.0,3,0,0,0,0,0,78.0,0.000092,86.36,81.50,0,88,False,False,False
1,2021,10,29,1,5,1.5,0.0,3,0,0,0,0,0,78.0,0.000099,86.90,82.40,0,88,False,False,False
2,2021,10,29,1,10,1.5,0.0,3,0,0,0,0,0,78.0,0.000123,88.16,85.46,0,88,False,False,False
3,2021,10,29,1,15,1.5,0.0,3,0,0,0,0,0,77.0,0.000127,88.70,86.18,0,88,False,False,False
4,2021,10,29,1,20,1.5,0.0,3,0,0,0,0,0,77.0,0.000098,89.06,87.08,0,88,False,False,False


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- meal_Snack


In [ ]:
# PREDICCIONS EN EL TEST
# S'ignoren els primers 60 minuts (12 files)
test = df_559_test.iloc[12:].reset_index(drop=True)

for h, fila in HORITZO.items():
    # Treiem les ultimes files perque no tindran prediccio
    Xt = test.iloc[:-fila].copy()
    preds = models[h].predict(Xt)
    out = pd.DataFrame({"idx_original": Xt.index, f"pred_glucose_t+{h}": preds})

    out.to_csv(f"../data/predicted/pred559_{h}min.csv", index=False)

    display(out.head())

### Evaluació del model

In [ ]:
metrics = []


for h, fila in HORITZO.items():
    # 1) y_true: glucosa real desplaçada -h
    y_true = test["glucose_level"].iloc[:-fila].reset_index(drop=True)
    
    # 2) y_pred: lee tu CSV de predicciones
    y_pred = (pd.read_csv(f"../data/predicted/pred559_{h}min.csv")[f"pred_glucose_t+{h}"].reset_index(drop=True))

    # 3) MAE
    mae = mean_absolute_error(y_true, y_pred)
    # 4) RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    metrics.append({
        "Pacient": 559,
        "Horitzo": f"{h} min",
        "RMSE": rmse,
        "MAE": mae
    })

result_df = pd.DataFrame(metrics)

result_df.loc["PROMIG"] = ["-", "-", result_df["RMSE"].mean(), result_df["MAE"].mean()]

print("\nResultats del pacient 559")
print(result_df)



Resultats del pacient 559
       Pacient Horitzo       RMSE        MAE
0          559  30 min  29.184779  22.158562
1          559  60 min  29.113176  22.093154
PROMIG       -       -  29.148978  22.125858
